In [0]:
from datetime import datetime
import uuid

from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)

EXECUTION_LOG_TABLE = "pipeline_execution_logs"
ERROR_LOG_TABLE = "pipeline_error_logs"
QUARANTINE_TABLE = "schema_drift_quarantine"

execution_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("batch_id", StringType(), True),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), True),
    StructField("parameter_or_file", StringType(), True),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", IntegerType(), False),
    StructField("rows_updated", IntegerType(), False),
    StructField("error_message", StringType(), True),
])

error_log_schema = StructType([
    StructField("error_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("batch_id", StringType(), True),
    StructField("layer", StringType(), False),
    StructField("source_file", StringType(), True),
    StructField("error_type", StringType(), False),
    StructField("error_message", StringType(), False),
    StructField("error_timestamp", TimestampType(), False),
])

for table_name, schema in [
    (EXECUTION_LOG_TABLE, execution_log_schema),
    (ERROR_LOG_TABLE, error_log_schema),
]:
    if not spark.catalog.tableExists(table_name):
        (
            spark.createDataFrame([], schema)
            .write.format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )
        print("Created:", table_name)
    else:
        print("Already exists:", table_name)

def log_execution(
    run_id,
    batch_id,
    layer,
    load_type,
    parameter_or_file,
    start_time,
    end_time,
    status,
    rows_inserted=0,
    rows_updated=0,
    error_message=None,
):
    record = [{
        "run_id": run_id,
        "batch_id": batch_id,
        "layer": layer,
        "load_type": load_type,
        "parameter_or_file": parameter_or_file,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_inserted": int(rows_inserted),
        "rows_updated": int(rows_updated),
        "error_message": error_message,
    }]

    (
        spark.createDataFrame(record, execution_log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(EXECUTION_LOG_TABLE)
    )

def log_error(run_id, batch_id, layer, source_file, error):
    record = [{
        "error_id": str(uuid.uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "layer": layer,
        "source_file": source_file,
        "error_type": type(error).__name__,
        "error_message": str(error),
        "error_timestamp": datetime.now(),
    }]

    (
        spark.createDataFrame(record, error_log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(ERROR_LOG_TABLE)
    )

print("Audit logging setup complete.")


Already exists: pipeline_execution_logs
Already exists: pipeline_error_logs
Audit logging setup complete.


In [0]:


success_run_id = str(uuid.uuid4())
start = datetime.now()
end = datetime.now()

log_execution(
    run_id=success_run_id,
    batch_id="logging_success_test",
    layer="Bronze-to-Silver",
    load_type="TEST",
    parameter_or_file="test batch",
    start_time=start,
    end_time=end,
    status="SUCCESS",
    rows_inserted=168,
    rows_updated=0,
)

failure_run_id = str(uuid.uuid4())
failure_start = datetime.now()

try:
    spark.read.json("/this/path/does/not/exist.json").count()
except Exception as e:
    failure_end = datetime.now()

    log_execution(
        run_id=failure_run_id,
        batch_id="forced_failure_test",
        layer="Raw-to-Bronze",
        load_type="TEST",
        parameter_or_file="/this/path/does/not/exist.json",
        start_time=failure_start,
        end_time=failure_end,
        status="FAILURE",
        rows_inserted=0,
        rows_updated=0,
        error_message=str(e),
    )

    log_error(
        run_id=failure_run_id,
        batch_id="forced_failure_test",
        layer="Raw-to-Bronze",
        source_file="/this/path/does/not/exist.json",
        error=e,
    )

print("SUCCESS and FAILURE logging tests completed.")

display(
    spark.table(EXECUTION_LOG_TABLE)
    .filter(col("run_id").isin(success_run_id, failure_run_id))
    .orderBy("start_time")
)


SUCCESS and FAILURE logging tests completed.


run_id batch_id layer load_type parameter_or_file start_time end_time status rows_inserted rows_updated error_message b75a4621-5da6-4b57-8151-2bb8175b3d7f logging_success_test Bronze-to-Silver TEST test batch 2026-10-06T15:24:09.336Z 2026-10-06T15:24:09.336Z SUCCESS 168 0 null bb49c10d-0681-432f-8123-cbf4711fccbc forced_failure_test Raw-to-Bronze TEST /this/path/does/not/exist.json 2026-10-06T15:24:12.065Z 2026-10-06T15:24:12.961Z FAILURE 0 0 [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /this/path/does/not/exist.json SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.getFileStatus(DatabricksFileSystemV2.scala:756)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.getFileStatus(DatabricksFileSystem.scala:215)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$4(LokiFileSystem.scala:710)
	at com.databricks.sql.io.LokiFileSystem.tryWithNativeIO(LokiFileSystem.scala:682)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$1(LokiFileSystem.scala:710)
	at com.databricks.sql.io.LokiFileSystem$.withWrappedExceptions(LokiFileSystem.scala:181)
	at com.databricks.sql.io.LokiFileSystem.getFileStatusNoCache(LokiFileSystem.scala:711)
	at com.databricks.sql.io.LokiFileSystem.getFileStatus(LokiFileSystem.scala:716)
	at org.apache.spark.deploy.SparkHadoopUtil$.isDirectory(SparkHadoopUtil.scala:768)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:70)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:568)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.$anonfun$loadV1BatchSource$3(ResolveDataSource.scala:400)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withPausedHangingDetection(QueryPlanningTracker.scala:325)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withPausedHangingDetection(QueryPlanningTracker.scala:317)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.$anonfun$loadV1BatchSource$2(ResolveDataSource.scala:400)
	at com.databricks.sql.managedcatalog.connections.FileSystemSourceConnectionUtils$.translateException(FileSystemSourceConnectionUtils.scala:164)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.$anonfun$applyOrElse$5(ResolveDataSource.scala:121)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:120)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:62)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$3(AnalysisHelper.scala:148)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$1(AnalysisHelper.scala:148)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper$.allowInvokingTransformsInAnalyzer(AnalysisHelper.scala:441)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsUpWithPruning(AnalysisHelper.scala:144)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsUpWithPruning$(AnalysisHelper.scala:140)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveOperatorsUpWithPruning(LogicalPlan.scala:49)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsUp(AnalysisHelper.scala:121)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHel

In [0]:

TEST_MERGE_TABLE = "audit_merge_metrics_test"

test_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("value", StringType(), True),
])

initial_df = spark.createDataFrame(
    [(1, "old_value"), (2, "keep_value")],
    test_schema,
)

(
    initial_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEST_MERGE_TABLE)
)

incoming_df = spark.createDataFrame(
    [(1, "new_value"), (3, "inserted_value")],
    test_schema,
)

run_id = str(uuid.uuid4())
start_time = datetime.now()

try:
    (
        DeltaTable.forName(spark, TEST_MERGE_TABLE)
        .alias("target")
        .merge(
            incoming_df.alias("source"),
            "target.id = source.id",
        )
        .whenMatchedUpdate(
            condition="NOT (target.value <=> source.value)",
            set={"value": "source.value"},
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    metrics = (
        DeltaTable.forName(spark, TEST_MERGE_TABLE)
        .history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
    )

    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated = int(metrics.get("numTargetRowsUpdated", 0))

    end_time = datetime.now()

    log_execution(
        run_id=run_id,
        batch_id="real_merge_metrics_test",
        layer="Silver-MERGE-Test",
        load_type="TEST",
        parameter_or_file=TEST_MERGE_TABLE,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        rows_inserted=rows_inserted,
        rows_updated=rows_updated,
    )

    print("Rows inserted:", rows_inserted)
    print("Rows updated:", rows_updated)

    assert rows_inserted == 1
    assert rows_updated == 1

except Exception as e:
    end_time = datetime.now()

    log_execution(
        run_id=run_id,
        batch_id="real_merge_metrics_test",
        layer="Silver-MERGE-Test",
        load_type="TEST",
        parameter_or_file=TEST_MERGE_TABLE,
        start_time=start_time,
        end_time=end_time,
        status="FAILURE",
        error_message=str(e),
    )

    log_error(
        run_id=run_id,
        batch_id="real_merge_metrics_test",
        layer="Silver-MERGE-Test",
        source_file=TEST_MERGE_TABLE,
        error=e,
    )

    raise

display(spark.table(TEST_MERGE_TABLE).orderBy("id"))


Rows inserted: 1
Rows updated: 1


id,value
1,new_value
2,keep_value
3,inserted_value


In [0]:

import json
import uuid
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType
)
from pyspark.sql.functions import col

QUARANTINE_TABLE = "schema_drift_quarantine"
TEST_VALID_TABLE = "schema_drift_valid_test"
TEST_BATCH_ID = "schema_drift_" + str(uuid.uuid4())

quarantine_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("raw_record", StringType(), False),
    StructField("failure_reason", StringType(), False),
    StructField("load_timestamp", TimestampType(), False),
])

valid_schema = StructType([
    StructField("city", StringType(), False),
    StructField("temperature_2m", DoubleType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

if not spark.catalog.tableExists(QUARANTINE_TABLE):
    (
        spark.createDataFrame([], quarantine_schema)
        .write.format("delta")
        .mode("overwrite")
        .saveAsTable(QUARANTINE_TABLE)
    )

print("Quarantine table ready:", QUARANTINE_TABLE)


# =====================================================
# 1. CREATE CONTROLLED SOURCE RECORDS
# =====================================================

raw_records = [
    '{"city":"Berlin","temperature_2m":20.5}',
    '{"city":"London","temperature_2m":18.2}',
    '{"city":"Paris","temperature_2m":"not_a_number"}',
    '{"city":"Rome","temperature_2m":25.4,"unexpected_field":"test"}'
]

print("Incoming records:", len(raw_records))


# =====================================================
# 2. AUTOMATIC VALIDATION
# =====================================================

expected_fields = {"city", "temperature_2m"}

valid_records = []
quarantine_records = []

def validate_record(raw_record):
    try:
        record = json.loads(raw_record)

        if not isinstance(record, dict):
            raise ValueError("Expected a JSON object")

        extra_fields = set(record.keys()) - expected_fields

        if extra_fields:
            raise ValueError(
                "Unexpected fields: " + ", ".join(sorted(extra_fields))
            )

        if not isinstance(record.get("city"), str) or not record["city"].strip():
            raise ValueError("Invalid or missing city")

        temperature = record.get("temperature_2m")

        if (
            isinstance(temperature, bool)
            or not isinstance(temperature, (int, float))
        ):
            raise ValueError(
                "temperature_2m must be numeric"
            )

        valid_records.append({
            "city": record["city"],
            "temperature_2m": float(temperature),
            "load_timestamp": datetime.now()
        })

    except (ValueError, TypeError, json.JSONDecodeError) as e:
        quarantine_records.append({
            "batch_id": TEST_BATCH_ID,
            "raw_record": raw_record,
            "failure_reason": str(e),
            "load_timestamp": datetime.now()
        })


for raw_record in raw_records:
    validate_record(raw_record)


# =====================================================
# 3. PROCESS VALID RECORDS
# =====================================================

valid_df = spark.createDataFrame(
    valid_records,
    valid_schema
)

(
    valid_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEST_VALID_TABLE)
)

print("Valid records processed:", valid_df.count())

display(spark.table(TEST_VALID_TABLE))


# =====================================================
# 4. QUARANTINE INVALID RECORDS
# =====================================================

if quarantine_records:
    quarantine_df = spark.createDataFrame(
        quarantine_records,
        quarantine_schema
    )

    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )

rejected_df = (
    spark.table(QUARANTINE_TABLE)
    .filter(col("batch_id") == TEST_BATCH_ID)
)

print("Quarantined records:", rejected_df.count())

display(rejected_df)


# =====================================================
# 5. VALIDATION
# =====================================================

valid_count = spark.table(TEST_VALID_TABLE).count()
quarantine_count = rejected_df.count()

assert len(raw_records) == 4
assert valid_count == 2
assert quarantine_count == 2
assert valid_count + quarantine_count == len(raw_records)

print("\n====================================")
print("SCHEMA DRIFT TEST PASSED")
print("Valid records:", valid_count)
print("Quarantined records:", quarantine_count)
print("Pipeline continued without bad records")
print("====================================")


Quarantine table ready: schema_drift_quarantine
Incoming records: 4
Valid records processed: 2


city,temperature_2m,load_timestamp
Berlin,20.5,2026-10-10T22:41:55.922Z
London,18.2,2026-10-10T22:41:55.922Z


Quarantined records: 2


batch_id,raw_record,failure_reason,load_timestamp
schema_drift_51bc8dca-99c9-4247-9e37-913346e64be0,"{""city"":""Paris"",""temperature_2m"":""not_a_number""}",temperature_2m must be numeric,2026-10-10T22:41:55.922Z
schema_drift_51bc8dca-99c9-4247-9e37-913346e64be0,"{""city"":""Rome"",""temperature_2m"":25.4,""unexpected_field"":""test""}",Unexpected fields: unexpected_field,2026-10-10T22:41:55.922Z



SCHEMA DRIFT TEST PASSED
Valid records: 2
Quarantined records: 2
Pipeline continued without bad records


In [0]:


required_tables = [
    "bronze_weather",
    "silver_weather",
    EXECUTION_LOG_TABLE,
    ERROR_LOG_TABLE,
    QUARANTINE_TABLE,
]

print("===== TABLE EXISTENCE =====")
for table_name in required_tables:
    print(
        table_name,
        "→",
        "PASS" if spark.catalog.tableExists(table_name) else "FAIL"
    )

silver_df = spark.table("silver_weather")
bronze_df = spark.table("bronze_weather")

silver_rows = silver_df.count()
distinct_keys = silver_df.select("city", "timestamp").distinct().count()
duplicates = silver_rows - distinct_keys
null_timestamps = silver_df.filter(col("timestamp").isNull()).count()
silver_null_load_ts = silver_df.filter(col("load_timestamp").isNull()).count()
bronze_null_load_ts = bronze_df.filter(col("load_timestamp").isNull()).count()

execution_logs = spark.table(EXECUTION_LOG_TABLE)
error_logs = spark.table(ERROR_LOG_TABLE)

success_logs = execution_logs.filter(col("status") == "SUCCESS").count()
failure_logs = execution_logs.filter(col("status") == "FAILURE").count()
quarantine_rows = spark.table(QUARANTINE_TABLE).count()

checks = {
    "Silver has no duplicate city + timestamp": duplicates == 0,
    "Silver has no null timestamps": null_timestamps == 0,
    "Bronze load_timestamp is present": bronze_null_load_ts == 0,
    "Silver load_timestamp is present": silver_null_load_ts == 0,
    "SUCCESS logging works": success_logs > 0,
    "FAILURE logging works": failure_logs > 0,
    "Detailed error logging works": error_logs.count() > 0,
    "Schema-drift quarantine works": quarantine_rows > 0,
}

print("\n===== FINAL CHECKLIST =====")
for name, passed in checks.items():
    print(name, "→", "PASS" if passed else "FAIL")

assert all(checks.values()), "One or more Phase 2 validation checks failed"

print("\nFINAL RESULT: ALL CHECKS PASSED")


===== TABLE EXISTENCE =====
bronze_weather → PASS
silver_weather → PASS
pipeline_execution_logs → PASS
pipeline_error_logs → PASS
schema_drift_quarantine → PASS

===== FINAL CHECKLIST =====
Silver has no duplicate city + timestamp → PASS
Silver has no null timestamps → PASS
Bronze load_timestamp is present → PASS
Silver load_timestamp is present → PASS
SUCCESS logging works → PASS
FAILURE logging works → PASS
Detailed error logging works → PASS
Schema-drift quarantine works → PASS

FINAL RESULT: ALL CHECKS PASSED
